# Multi-Agent Collaborative Code Generation — Evaluation
## 5 Baselines + Ours on HumanEval Test Set

B1: Single Qwen3-0.6B  |  B3: Parallel 2×0.6B  |  B4: Sequential
B2: Single Coder-1.5B  |  B5: Discussion  |   Ours: IAC 2×0.6B

In [ ]:
!pip install -q transformers datasets accelerate tqdm pyyaml

In [ ]:
import os, sys, torch

REPO = "https://github.com/JKpink/CoMLRL.git"
DIR = "CoMLRL"

if not os.path.exists(DIR):
    !git clone {REPO}
else:
    %cd {DIR}
    !git pull origin main
    %cd ..

%cd {DIR}
!pip install -q -e . --no-deps
sys.path.insert(0, "examples")

In [ ]:
import ast, contextlib, io, re, signal
from typing import List
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

from code_collab import cleanup_code, extract_test_cases

MODEL = "Qwen/Qwen3-0.6B"
CODER_MODEL = "Qwen/Qwen2.5-Coder-1.5B"
test_data = load_dataset("openai_humaneval", trust_remote_code=True, split="test").select(range(100, 164))
print(f"Test set: {len(test_data)} problems")

In [ ]:
def run_single(model_name, problem):
    """B1/B2: Single model does everything."""
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16,
    )
    prompt = problem["prompt"]
    full = f"Complete this Python function:\n{prompt}\n\nYour code:"
    inputs = tokenizer(full, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=False)
    code = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return score_code(problem["prompt"] + "\n" + cleanup_code(code), problem)

def run_sequential(base_model, problem):
    """B3/B4: Sequential A→B with base model."""
    from code_collab import helper_formatter, main_formatter
    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        base_model, device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16,
    )
    # A generates
    pa = helper_formatter(problem)
    ia = tokenizer(pa, return_tensors="pt").to(model.device)
    with torch.no_grad():
        oa = model.generate(**ia, max_new_tokens=128, temperature=0.1, do_sample=False)
    ca = tokenizer.decode(oa[0][ia.input_ids.shape[1]:], skip_special_tokens=True)
    # B generates
    pb = main_formatter(problem)
    ib = tokenizer(pb, return_tensors="pt").to(model.device)
    with torch.no_grad():
        ob = model.generate(**ib, max_new_tokens=256, temperature=0.1, do_sample=False)
    cb = tokenizer.decode(ob[0][ib.input_ids.shape[1]:], skip_special_tokens=True)
    return score_code(cleanup_code(ca) + "\n\n" + cleanup_code(cb), problem)

def score_code(code, problem):
    """Score: 0-1 based on syntax + execution."""
    s = 0.0
    try:
        ast.parse(code); s += 0.5
        with contextlib.redirect_stdout(io.StringIO()): exec(code, {}); s += 0.5
    except: pass
    return s

print("Evaluation helpers loaded")

In [ ]:
# Run all baselines (on a subset for speed)
N = min(20, len(test_data))
results = {}

print("=== B1: Single Qwen3-0.6B ===")
results["B1_Single_0.6B"] = [run_single(MODEL, test_data[i]) for i in tqdm(range(N))]

print("=== B2: Single Coder-1.5B ===")
results["B2_Single_Coder"] = [run_single(CODER_MODEL, test_data[i]) for i in tqdm(range(N))]

print("=== B3: Sequential 2×0.6B ===")
results["B3_Sequential"] = [run_sequential(MODEL, test_data[i]) for i in tqdm(range(N))]

print("=== Ours: IAC 2×0.6B ===")
# Load trained IAC weights from /kaggle/working/outputs/iac_code
if os.path.exists("/kaggle/working/outputs/iac_code"):
    print("Loading trained weights...")
    # (use IACTrainer.evaluate or manual inference)
    results["Ours_IAC"] = ["TODO: load model and run"] * N
else:
    print("No trained weights found — run training notebook first")
    results["Ours_IAC"] = [0] * N

# Summary
print("\n" + "=" * 50)
for method, scores in results.items():
    avg = sum(scores) / len(scores) if scores else 0
    print(f"{method:<25} avg={avg:.3f}")